# Train TinySAM from scratch on Colab

This notebook uses a small, explainable model rather than reproducing the original EfficientSAM scale. The original SAMI stage distills from a frozen SAM ViT-H teacher on ImageNet-1K, then fine-tunes on SA-1B; this project has no pretrained teacher and uses Pets with pixel MAE pretraining. Select **Runtime → Change runtime type → T4 GPU** for the full 256-pixel PyTorch run. The CPU smoke cells near the end can be used on a laptop.


## 1. Check the runtime and clone this branch

The branch variable is set to the branch used for this implementation. If you copied the notebook from another branch, update `BRANCH` before running the cell.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

REPO_URL = 'https://github.com/Mehdi-blip-stack/EfficientSAM'
BRANCH = 'devin/1792273248-from-scratch-tinysam'
!git clone --branch {BRANCH} {REPO_URL} /content/EfficientSAM
%cd /content/EfficientSAM/from_scratch

## 2. Install the small project's dependencies

In [ ]:
%pip install -r requirements.txt

## 3. Download and cache Oxford-IIIT Pets

The original archives are large and cached locally. Do not commit `data/`.

In [ ]:
!python scripts/download_data.py --size 256

## 4. Check the hand-written derivatives

The finite-difference tests use float64 and compare both individual layers and the full NumPy model against centered numerical derivatives. The PyTorch parity test is an independent check of all parameter gradients.

In [ ]:
!pytest tests -q

## 5. Pretrain the encoder with masked pixel reconstruction

For each batch, 75% of patches are hidden. The visible patches pass through the encoder, and a small decoder reconstructs normalized patch pixels; loss is calculated only on hidden patches. This is not SAMI feature distillation: we do not use a pretrained teacher.

In [ ]:
!python scripts/pretrain_mae_torch.py --steps 1000 --batch-size 32 --device auto --out runs/mae_encoder.pt

## 6. Train promptable segmentation

Training uses focal + soft Dice on the best of three mask tokens and MSE for the predicted IoU. The `--max-steps` and `--steps-per-epoch` flags are useful for a quick check; remove them or increase them for a longer run.

In [ ]:
!python scripts/train_torch.py --init-encoder runs/mae_encoder.pt --device auto --epochs 20 --batch-size 32 --name pets-torch

## 7. Evaluate deterministic point and box prompts

Point evaluation uses the foreground pixel with the largest two-pass chamfer distance to the background. Box evaluation uses the exact foreground bounding box. Each result is saved as JSON next to the weights.

In [ ]:
!python scripts/evaluate.py --impl torch --weights runs/pets-torch/weights.pt --prompt point
!python scripts/evaluate.py --impl torch --weights runs/pets-torch/weights.pt --prompt box

## 8. Inspect predictions and the training curve

In [ ]:
!python scripts/visualize.py --impl torch --weights runs/pets-torch/weights.pt --output runs/pets-torch/visualize.png
from IPython.display import Image, display
display(Image('runs/pets-torch/visualize.png'))

import csv
import matplotlib.pyplot as plt
with open('runs/pets-torch/loss.csv') as file:
    losses = list(csv.DictReader(file))
plt.plot([int(row['step']) for row in losses], [float(row['loss']) for row in losses])
plt.xlabel('Training step'); plt.ylabel('Loss'); plt.grid(True); plt.show()

## 9. Download the trained weights

The ZIP includes the torch checkpoint and measured curves/metrics, not the dataset.

In [ ]:
!zip -r tinysam-results.zip runs/pets-torch/weights.pt runs/pets-torch/loss.csv runs/pets-torch/val_metrics.csv runs/pets-torch/loss_curve.png runs/pets-torch/visualize.png
from google.colab import files
files.download('tinysam-results.zip')

## CPU smoke-test commands

These use the same scripts with a 64-pixel, one-block model. The implementation should run on CPU; this smoke model is intentionally not a replacement for the 256-pixel training above.

In [ ]:
!python scripts/pretrain_mae_torch.py --tiny --size 64 --patch 8 --steps 20 --batch-size 2 --device cpu --out runs/smoke_mae.pt
!python scripts/train_torch.py --tiny --size 64 --patch 8 --max-steps 50 --steps-per-epoch 50 --epochs 1 --batch-size 32 --warmup-steps 5 --lr 1e-3 --device cpu --name smoke-verified --val-subset 8
!python scripts/evaluate.py --impl torch --tiny --size 64 --patch 8 --max-images 4 --weights runs/smoke-verified/weights.pt --prompt point